# Feature selection

Pipeline autonome reprise de `features_engineering.ipynb` : **multivariate_4_history + all_groups + HistGB optimisé**
La référence sans sélection est recalculée ci-dessous sur cinq splits patients.
Les features `all_groups` sont calculées après l’imputation des variables d’origine.
La sélection intervient après tous les prétraitements, juste avant le modèle.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from threadpoolctl import threadpool_limits

DATA_DIR = Path("data") if Path("data").is_dir() else Path("../data")
X = pd.read_csv(DATA_DIR / "X_train_6ZIKlTY.csv")
Y = pd.read_csv(DATA_DIR / "y_train_lXj6X5y.csv")
assert X["Index"].equals(Y["Index"])
y = Y["target"]


## Historique patient et imputation


In [2]:
# Names of helper features created for the imputation regressors.
diagnosis_history = [
    "ledd_first_observed", "time_to_first_observed_ledd", "time_since_first_visit",
]
history_features = diagnosis_history + [
    f"{col}_{stat}_patient" for col in ["on", "off", "ledd"] for stat in ["median", "slope"]
] + ["disease_duration"]

def median_other_visits(s):
    values = np.tile(s.to_numpy(), (len(s), 1))
    # Score/dose summaries exclude the current visit.
    np.fill_diagonal(values, np.nan)
    available = ~np.isnan(values).all(axis=1)
    medians = np.full(len(s), np.nan)
    medians[available] = np.nanmedian(values[available], axis=1)
    return medians

def add_patient_history(df):
    visits = df.sort_values(["patient_id", "age"])
    first_age = visits.groupby("patient_id")["age"].min()
    first_dose = visits.dropna(subset=["ledd"]).drop_duplicates("patient_id").set_index("patient_id")
    out = df.copy()
    out["ledd_first_observed"] = out["patient_id"].map(first_dose["ledd"])
    out["time_to_first_observed_ledd"] = out["patient_id"].map(first_dose["age"]) - out["patient_id"].map(first_age)
    out["time_since_first_visit"] = out["age"] - out["patient_id"].map(first_age)
    out["disease_duration"] = out["age"] - out["age_at_diagnosis"]
    t = out["time_since_first_visit"]
    for col in ["on", "off", "ledd"]:
        v = df[col]
        out[f"{col}_median_patient"] = df.groupby("patient_id")[col].transform(median_other_visits)
        sums = pd.DataFrame({"n": v.notna().astype(int), "t": t.where(v.notna(), 0),
                             "v": v.fillna(0), "tt": (t*t).where(v.notna(), 0), "tv": (t*v).fillna(0)})
        # Subtract this visit's contributions from the patient's regression sums.
        others = sums.groupby(df["patient_id"]).transform("sum") - sums
        denominator = others["n"] * others["tt"] - others["t"]**2
        out[f"{col}_slope_patient"] = (
            (others["n"] * others["tv"] - others["t"] * others["v"]) / denominator
        ).where((others["n"] >= 3) & (denominator > 1e-12))
    return out


In [3]:
encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
      ["cohort", "gene"])],
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")

preprocessor = Pipeline([
    ("history", FunctionTransformer(add_patient_history)),
    ("encoding", encoder),
    ("remove_ids", FunctionTransformer(
        lambda df: df.drop(columns=["Index"])
    )),
])

predictors_history = {
    "age_at_diagnosis": ["age"] + diagnosis_history,
    "on": ["off", "age_at_diagnosis", "on_median_patient",
           "off_median_patient", "ledd_median_patient", "disease_duration",
           "time_since_first_visit", "on_slope_patient"],
    "off": ["on", "age_at_diagnosis", "time_since_intake_on", "on_median_patient",
            "off_median_patient", "ledd_median_patient",
            "time_since_first_visit", "off_slope_patient"],
    "ledd": ["age", "age_at_diagnosis", "on", "cohort_A",
             "off_median_patient", "ledd_median_patient", "disease_duration",
             "time_since_first_visit", "ledd_slope_patient"],
}

class PredictiveImputer(TransformerMixin, BaseEstimator):
    def __init__(self, predictors):
        self.predictors = predictors

    def fit(self, X, y=None):
        self.medians_, self.models_ = X.median(numeric_only=True), {}
        self.indicator_ = MissingIndicator(error_on_new=False).set_output(
            transform="pandas"
        ).fit(X.drop(columns=history_features))
        for col, features in self.predictors.items():
            observed = X[col].notna()
            self.models_[col] = HistGradientBoostingRegressor(
                max_iter=50, max_leaf_nodes=15, random_state=42
            ).fit(X.loc[observed, features], X.loc[observed, col])
        return self

    def transform(self, X):
        filled = X.copy()
        for col, features in self.predictors.items():
            missing = X[col].isna()
            if missing.any():
                values = self.models_[col].predict(X.loc[missing, features])
                if col == "age_at_diagnosis":  # Diagnosis cannot follow this visit.
                    values = values.clip(max=X.loc[missing, "age"].to_numpy())
                filled.loc[missing, col] = values
        if "disease_duration" in filled:
            filled["disease_duration"] = filled["age"] - filled["age_at_diagnosis"]
        # History supports imputation only; indicators describe the original missingness.
        values = filled.fillna(self.medians_).drop(columns=history_features)
        indicators = self.indicator_.transform(X.drop(columns=history_features)).astype(float)
        return pd.concat([values, indicators], axis=1)

imputation_pipeline = make_pipeline(PredictiveImputer(predictors_history))


## Features retenues : `all_groups`


In [4]:
# Features calculées depuis les variables d’origine après imputation.
def build_visit_features(data):
    patients = data.groupby("patient_id")
    ages = patients["age"]
    features = pd.DataFrame(index=data.index)

    # Temps écoulé et visites strictement antérieures/ultérieures (hors âges identiques).
    features["time_since_first_visit"] = data["age"] - ages.transform("min")
    features["n_past_visits"] = (ages.rank(method="min") - 1).astype(int)
    features["n_future_visits"] = (ages.transform("size") - ages.rank(method="max")).astype(int)
    for col in ["on", "off", "ledd"]:
        # Médiane de toutes les mesures observées, visite actuelle comprise.
        values = data[col]
        features[f"{col}_median_patient"] = patients[col].transform("median")

        # Pente annuelle de la droite ajustée : au moins deux âges observés distincts.
        age = data["age"].where(values.notna())
        dt = age - age.groupby(data["patient_id"]).transform("mean")
        dv = values - patients[col].transform("mean")
        covariance = (dt * dv).groupby(data["patient_id"]).transform("sum")
        variance = (dt**2).groupby(data["patient_id"]).transform("sum")
        features[f"{col}_slope_patient"] = (covariance / variance).where(variance > 1e-12)
    gap = data["off"] - data["on"]  # NaN si un des deux scores manque.
    features["off_on_median_patient"] = gap.groupby(data["patient_id"]).transform("median")
    features["disease_duration"] = data["age"] - data["age_at_diagnosis"]
    # Interactions dose/délai, calculées après imputation.
    for score in ["on", "off"]:
        t = data[f"time_since_intake_{score}"]
        features[f"ledd_x_time_{score}"] = data["ledd"] * t
        features[f"ledd_time_ratio_{score}"] = data["ledd"] / (1 + t)

    # Niveau de la droite du patient à l'âge actuel (visite actuelle comprise).
    for col in ["on", "off"]:
        mean_age = data["age"].where(data[col].notna()).groupby(data["patient_id"]).transform("mean")
        features[f"{col}_trend_at_visit"] = patients[col].transform("mean") + (
            features[f"{col}_slope_patient"] * (data["age"] - mean_age)
        )

    # Mesures observées voisines, en sautant les NaN et en excluant la visite actuelle.
    visits = data.sort_values(["patient_id", "age"])
    for col in ["on", "off"]:
        observed = visits[[col, "age"]].where(visits[col].notna(), axis=0)
        for label, shift, fill in [("previous", 1, "ffill"), ("next", -1, "bfill")]:
            neighbor = getattr(observed.groupby(visits["patient_id"]), fill)()
            neighbor = neighbor.groupby(visits["patient_id"]).shift(shift)
            features[f"{col}_{label}"] = neighbor[col]
            features[f"{col}_{label}_gap"] = (visits["age"] - neighbor["age"]).abs()

    # Nombre de mesures observées et dispersion par patient.
    for col in ["on", "off", "ledd"]:
        features[f"{col}_n_observed"] = patients[col].transform("count")
    for col in ["on", "off"]:
        features[f"{col}_std_patient"] = patients[col].transform("std")
    return features


## Pipeline complète

`pipeline` accepte les données brutes et peut être ajustée avec `clone(pipeline).fit(X_train, y_train)`.
Remplacer l’étape `selection` avec `pipeline.set_params(selection=votre_selecteur)`.
Les sorties du preprocessing sont des DataFrames avec les noms de features.


In [5]:
features_eng = FunctionTransformer(
    lambda df: df.join(build_visit_features(df)).drop(columns=["patient_id"])
)

best_params = {
    "learning_rate": 0.042592090635273853, "max_iter": 800,
    "max_leaf_nodes": 62, "min_samples_leaf": 25,
    "l2_regularization": 1.8115270802885095, "max_features": 0.6078232265551354,
}
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("imputation", imputation_pipeline),
    ("features_eng", features_eng),
    ("feature_imputation", SimpleImputer(
        strategy="median", add_indicator=True, keep_empty_features=True
    ).set_output(transform="pandas")),
    ("selection", "passthrough"),
    ("model", HistGradientBoostingRegressor(random_state=42, **best_params)),
])


## Préparer les cinq splits une seule fois

Tous les prétraitements appris sont ajustés uniquement sur le train du split.
Les patients sont séparés ; leurs features utilisent les visites disponibles dans leur propre ensemble.
Les folds préparés peuvent être réutilisés tant que les splits et le preprocessing ne changent pas.


In [6]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        fitted = clone(pipeline[:-2])
        prepared_folds.append((
            fitted.fit_transform(X.iloc[train_idx], y.iloc[train_idx]),
            fitted.transform(X.iloc[val_idx]), y.iloc[train_idx], y.iloc[val_idx],
        ))


## Essais de sélection

`evaluate_selection()` donne la référence sans sélection. Passer ensuite un sélecteur scikit-learn à cette fonction :
il sera cloné et ajusté uniquement sur le train de chaque split, avant HistGB.
Seuls le sélecteur et le modèle sont réentraînés à chaque essai ; l’imputation reste en cache dans `prepared_folds`.


In [7]:
def evaluate_selection(selector="passthrough"):
    candidate = clone(pipeline[-2:]).set_params(selection=selector)
    scores = []
    with threadpool_limits(limits=3):
        for train, validation, yt, yv in prepared_folds:
            fitted = clone(candidate).fit(train, yt)
            scores.append(mean_squared_error(yv, fitted.predict(validation)))
    return pd.Series({"MSE": np.mean(scores), "std": np.std(scores)})

baseline = evaluate_selection()
baseline


MSE    11.922491
std     0.340410
dtype: float64

In [8]:
from functools import partial
from tempfile import TemporaryDirectory
from joblib import Memory
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.preprocessing import StandardScaler

# f_regression est un score pour SelectKBest ; l'autre variante utilise l'information mutuelle.
mi_cache = TemporaryDirectory(prefix="parkinson_selection_")
mi_score = Memory(mi_cache.name, verbose=0).cache(mutual_info_regression)
selectors = {
    "PCA": lambda k: make_pipeline(StandardScaler(), PCA(n_components=k)),
    "SelectKBest_MI": lambda k: SelectKBest(partial(mi_score, random_state=42), k=k),
    "SelectKBest_f_regression": lambda k: SelectKBest(f_regression, k=k),
}
max_features = min(train.shape[1] for train, *_ in prepared_folds)
selection_results = pd.DataFrame([
    {"method": name, "k": k, **evaluate_selection(build(k))}
    for name, build in selectors.items() for k in range(8, max_features + 1, 2)
])
selection_results["delta_MSE"] = selection_results["MSE"] - baseline["MSE"]
selection_results.sort_values("MSE")


,method,k,MSE,std,delta_MSE
69,SelectKBest_f_regression,42,11.866788,0.275643,-0.055703
77,SelectKBest_f_regression,58,11.870570,0.271128,-0.051921
76,SelectKBest_f_regression,56,11.879836,0.330131,-0.042655
51,SelectKBest_MI,58,11.905894,0.302930,-0.016597
74,SelectKBest_f_regression,52,11.908525,0.327369,-0.013966
50,SelectKBest_MI,56,11.924520,0.347080,0.002029
48,SelectKBest_MI,52,11.936321,0.318084,0.013830
75,SelectKBest_f_regression,54,11.949189,0.301417,0.026698
49,SelectKBest_MI,54,11.950461,0.314019,0.027970
47,SelectKBest_MI,50,11.974993,0.233030,0.052502
